In [0]:
%pip install sqlglot
%restart_python

In [0]:
%sql
CREATE OR REPLACE TABLE  dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition 
as (
    WITH API AS (
    SELECT
        DataItem_unique_Id,
        universe_id,
        DataItem_id,
        DataItem_name,
        case 
          when endswith(universe_name, '.unx') then left(universe_name, length(universe_name) - 4) 
          when endswith(universe_name, '_old_udt') then left(universe_name, length(universe_name) - 8) 
          when universe_name = 'Dallas' then 'Dallas2'
          when universe_name = 'Combined Paym''t & Recovery' then 'Combined Paym''t _ Recovery'
          else universe_name 
        end as universe_name,
        DataItem_type,
        DataItem_description,
        DataItem_dataType,
        -- Process DataItem_path: split by '\', remove everything after '|' in each part, then join back with '\'
        concat(
        '\\',
        array_join(
            transform(
            slice(
                split(DataItem_path, '\\\\'),
                1,
                size(split(DataItem_path, '\\\\')) - 1
            ),
            part -> split(part, '\\|')[0]
            ),
            '\\'
        )
        ) AS DataItem_path,
        universe_path,
        universe_type
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_datafields_api_list
    WHERE universe_name NOT LIKE '%Obsolete%'
    ),
    idt_by_id AS (
    SELECT
        universe_name,
        Cuid AS DataItem_id,
        Name AS DataItem_name,
        `Select` AS sql_definition,
        Path AS DataItem_path,
        ROW_NUMBER() OVER (PARTITION BY universe_name, Cuid ORDER BY CASE WHEN `Select` IS NOT NULL THEN 0 ELSE 1 END) as rn
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_unvx_definitions
    ),
    idt_by_name AS (
    SELECT
        universe_name,
        Name AS DataItem_name,
        `Select` AS sql_definition,
        Path AS DataItem_path,
        ROW_NUMBER() OVER (PARTITION BY universe_name, Name, Path ORDER BY CASE WHEN `Select` IS NOT NULL THEN 0 ELSE 1 END, Cuid) as rn
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_unvx_definitions
    )
    select API.*, 
        COALESCE(id_match.sql_definition, name_match.sql_definition) as sql_definition,
        COALESCE(id_match.DataItem_name, name_match.DataItem_name) as DataItem_name_definition,
        COALESCE(
            left(id_match.DataItem_path, length(id_match.DataItem_path) - 1),
            left(name_match.DataItem_path, length(name_match.DataItem_path) - 1)
        ) as DataItem_path_definition
    from API 
    left join idt_by_id id_match
        on upper(api.universe_name) = upper(id_match.universe_name)
        and API.DataItem_id = id_match.DataItem_id
        and id_match.rn = 1
    left join idt_by_name name_match
        on id_match.DataItem_id IS NULL
        and upper(api.universe_name) = upper(name_match.universe_name)
        and upper(API.DataItem_name) = upper(name_match.DataItem_name)
        and upper(API.DataItem_path) = upper(left(name_match.DataItem_path, length(name_match.DataItem_path) - 1))
        and name_match.rn = 1
)

In [0]:
%sql
SELECT 
    universe_name,
    COUNT(*) AS total_api_fields,
    SUM(CASE WHEN sql_definition IS NOT NULL THEN 1 ELSE 0 END) AS matched_idt,
    SUM(CASE WHEN sql_definition IS NULL THEN 1 ELSE 0 END) AS unmatched,
    ROUND(100.0 * SUM(CASE WHEN sql_definition IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS match_pct
FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition
GROUP BY universe_name
ORDER BY match_pct DESC, universe_name


Usage Note: 
Universe ID matching is not reliable, better to use Universe Name remove **.unx & _old_ud**t
-     AM Tool Universe
-     AM Tool Universe.unx
-     AM Tool Universe_old_udt

**_Data Item matching need to combine path_**

In [0]:

import re
from pyspark.sql.functions import udf, expr
from pyspark.sql.types import ArrayType, StringType

df = spark.table("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition")

# Extract all TABLE.COLUMN or SCHEMA.TABLE.COLUMN references
# Strips away functions like SUM(), COUNT(), DECODE(), NVL(), CASE, TO_DATE(), @Prompt() etc.
def extract_table_refs(sql_def):
    if not sql_def:
        return None
    # Strip single-quoted string literals to avoid false positives (e.g. 'N.A', 'DD.MM.YYYY')
    cleaned = re.sub(r"'[^']*'", '', sql_def)
    # Match dotted identifiers: 2+ parts separated by dots
    pattern = r'[A-Za-z_][A-Za-z0-9_#$]*\.[A-Za-z_][A-Za-z0-9_#$]*(?:\.[A-Za-z_][A-Za-z0-9_#$]*)*'
    matches = re.findall(pattern, cleaned)
    # Deduplicate while preserving order
    seen = set()
    result = []
    for m in matches:
        if m not in seen:
            seen.add(m)
            result.append(m)
    return result if result else None

extract_table_refs_udf = udf(extract_table_refs, ArrayType(StringType()))

# Extract table name from each ref by taking everything before the last dot
def get_table_names(refs):
    if not refs:
        return None
    tables = []
    seen = set()
    for ref in refs:
        table = ref.rsplit('.', 1)[0]  # everything before last dot
        if table and table not in seen:
            seen.add(table)
            tables.append(table)
    return tables if tables else None

extract_table_refs_udf = udf(extract_table_refs, ArrayType(StringType()))
get_table_names_udf = udf(get_table_names, ArrayType(StringType()))

df_with_refs = df.withColumn("table_column_refs", extract_table_refs_udf(df["sql_definition"])) \
    .withColumn("SQL_Tables", get_table_names_udf("table_column_refs")) \
    .withColumn("SQL_Tables_str", expr("array_join(SQL_Tables, ' | ')"))

# Keep all original columns + SQL_Tables_str, write back to the table
df_final = df_with_refs.select(*df.columns, "SQL_Tables_str")
df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition")


In [0]:
%sql

select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.universe_dataentry_definition
where universe_name='Overdues Claims BSB'